# T-DEED в Colab (Football AI Coach)

Тонкая обёртка над командой `aicoach tdeed`: ноутбук ничего не считает сам. Он ставит проект, берёт видео с Google Drive, запускает команду на видеокарте и возвращает JSON на Drive.

**Перед запуском:** `Среда выполнения → Сменить тип среды выполнения → GPU (T4)`. Заполните ячейку «Параметры» и выполняйте ячейки по порядку.

torch, torchvision и numpy берутся предустановленные из Colab, проект их не переустанавливает.

Для проверки отрезка оставьте `FROM` и `TO` (по умолчанию 10 минут 1-го тайма); для всего матча очистите оба поля. Отрезок пишется в отдельный файл `tdeedseg_*.json`, готовый `tdeed_*.json` ноутбук не перезаписывает.


In [ ]:
# Параметры: меняйте только эту ячейку.
REPO_URL = "https://github.com/ddanakr/football-ai-coach.git"  # ссылка на репозиторий
BRANCH = "main"  # ветка репозитория
VIDEO_ON_DRIVE = "/content/drive/MyDrive/football/match01/src.mp4"  # путь к src.mp4 на Drive
FROM = "13:12"  # начало отрезка (мм:сс или чч:мм:сс); "" и "" в обоих полях: весь матч
TO = "23:12"  # конец отрезка; 10 минут от начала 1-го тайма (подставьте свои времена)


In [ ]:
# Проверка видеокарты и версии Python.
import sys

!nvidia-smi
print("Python", sys.version.split()[0])


In [ ]:
# Клонирование репозитория проекта.
!rm -rf /content/football-ai-coach
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/football-ai-coach


In [ ]:
# Установка проекта и зависимостей T-DEED. torch, torchvision и numpy предустановлены в Colab
# и не переустанавливаются: в [gpu] их нет.
from importlib import metadata


def versions():
    found = {}
    for name in ("torch", "torchvision", "numpy", "timm"):
        try:
            found[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            found[name] = "нет"
    return found


before = versions()
%cd /content/football-ai-coach
!pip install -q ".[gpu]"
after = versions()
print("До установки:", before)
print("После:", after)
changed = [n for n in before if before[n] != "нет" and before[n] != after[n]]
if changed:
    print("Изменились", changed, ": Среда выполнения -> Перезапустить сеанс, затем выполните ячейки с начала.")


In [ ]:
# Проверка: проект импортируется, версии библиотек, видеокарта.
import aicoach
import numpy
import timm
import torch

print("aicoach:", aicoach.__file__)
print("torch", torch.__version__, "| numpy", numpy.__version__, "| timm", timm.__version__)
print("CUDA доступна:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Нет видеокарты: выберите среду выполнения с GPU и начните сначала"


In [ ]:
# Подключение Google Drive и копирование видео на локальный диск Colab (так быстрее).
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
video_on_drive = Path(VIDEO_ON_DRIVE)
assert video_on_drive.is_file(), f"На Drive нет файла {video_on_drive}"
MATCH_LOCAL = Path("/content/match")
MATCH_LOCAL.mkdir(exist_ok=True)
if not (MATCH_LOCAL / "src.mp4").exists():
    shutil.copy(video_on_drive, MATCH_LOCAL / "src.mp4")
print("Видео:", (MATCH_LOCAL / "src.mp4").stat().st_size // 2**20, "МБ")


In [ ]:
# Запуск: код T-DEED скачивается в кэш, чекпойнт тоже, затем идёт инференс на видеокарте.
SEGMENT_ARGS = f"--from {FROM} --to {TO}" if FROM and TO else ""
%cd /content/football-ai-coach
!python -m aicoach.cli tdeed --match-dir {MATCH_LOCAL} --device cuda {SEGMENT_ARGS}


In [ ]:
# Копирование результата на Drive. Существующий файл не перезаписывается:
# новый кладётся рядом с суффиксом .colab (его не подхватит ни одна стадия).
import shutil

target_dir = video_on_drive.parent
for found in sorted(MATCH_LOCAL.glob("tdeed*.json")):
    destination = target_dir / found.name
    if destination.exists():
        destination = target_dir / (found.name + ".colab")
    shutil.copy(found, destination)
    print("Скопировано:", destination.name)


In [ ]:
# Время стадии из status.json (то же печатает команда: на минуту видео и оценка на матч).
import json

status = json.loads((MATCH_LOCAL / "status.json").read_text(encoding="utf-8"))
for name in ("tdeed", "tdeed_segment"):
    entry = status["stages"].get(name)
    if entry:
        print(name, entry["state"], entry["duration_s"], "с")
